In [1]:
"""
Frame processing functions for the motion detection project.
"""

import cv2
import numpy as np

def process_video(video_path, target_fps=5, resize_dim=(1280, 720)):
    """
    Extract frames from a video at a specified frame rate.

    Args:
        video_path: Path to the video file
        target_fps: Target frames per second to extract
        resize_dim: Dimensions to resize frames to (width, height)

    Returns:
        List of extracted frames
    """
    # Open the video file
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video file: {video_path}")

    # Get video properties
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print("original fps :",original_fps, " & frame count :", frame_count)

    # Calculate frame interval for the target FPS
    frame_interval = max(1, int(round(original_fps / target_fps)))
    print("frame interval:", frame_interval)

    # TODO: Implement frame extraction
    # 1. Read frames from the video capture object
    # 2. Only keep frames at the specified interval to achieve target_fps
    # 3. Resize frames to the specified dimensions
    # 4. Store frames in a list
    # 5. Release the video capture object when done

    # Example starter code:
    frames = []
    frame_index = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_index % (frame_interval) == 0: # denominator is increased by 1 to get the target fps and frame count
            frame = cv2.resize(src=frame,dsize=resize_dim,interpolation=cv2.INTER_AREA) # cv2.INTER_AREA is good for shrinking images.
            frames.append(frame)        
        frame_index += 1
    
    cap.release()
    # Your implementation here

    return frames

In [2]:
video_path = "E:\\New Application\\HomeTeam\\viewport_tracking\\hometeam_ai_assignment\\sample_video_clip.mp4"
frames = process_video(video_path=video_path)


original fps : 29.97002997002997  & frame count : 416
frame interval: 6


In [3]:
len(frames)

50

In [4]:
# motion_detector.py
"""
Motion detection functions for the sports video analysis project.
"""

import cv2
import numpy as np


def detect_motion(frames, frame_idx, threshold=25, min_area=100):
    """
    Detect motion in the current frame by comparing with previous frame.

    Args:
        frames: List of video frames
        frame_idx: Index of the current frame
        threshold: Threshold for frame difference detection
        min_area: Minimum contour area to consider

    Returns:
        List of bounding boxes for detected motion regions
    """
    # We need at least 2 frames to detect motion
    if frame_idx < 1 or frame_idx >= len(frames):
        return []

    # Get current and previous frame
    current_frame = frames[frame_idx]
    prev_frame = frames[frame_idx - 1]

    # TODO: Implement motion detection
    # 1. Convert frames to grayscale
    # 2. Apply Gaussian blur to reduce noise (hint: cv2.GaussianBlur)
    # 3. Calculate absolute difference between frames (hint: cv2.absdiff)
    # 4. Apply threshold to highlight differences (hint: cv2.threshold)
    # 5. Dilate the thresholded image to fill in holes (hint: cv2.dilate)
    # 6. Find contours in the thresholded image (hint: cv2.findContours)
    # 7. Filter contours by area and extract bounding boxes

    # Example starter code:
    motion_boxes = []

    # Your implementation here
    current_frame_gray = cv2.cvtColor(current_frame,cv2.COLOR_BGR2GRAY)
    prev_frame_gray = cv2.cvtColor(prev_frame,cv2.COLOR_BGR2GRAY)
    current_frame_gray_blurred = cv2.GaussianBlur(current_frame_gray,(5,5),0)
    prev_frame_gray_blurred = cv2.GaussianBlur(prev_frame_gray,(5,5),0)
    diff_gray = cv2.absdiff(prev_frame_gray_blurred,current_frame_gray_blurred)
    _, thresh = cv2.threshold(diff_gray,threshold,255,cv2.THRESH_BINARY)
    dilated = cv2.dilate(thresh,None,iterations=3)
    contours,_ = cv2.findContours(dilated, cv2.RETR_TREE,cv2.CHAIN_APPROX_SIMPLE)
    #print("contours :",contours)
    
    img = dilated
    for contour in contours:
        (x,y,w,h) = cv2.boundingRect(contour)
        if cv2.contourArea(contour) < min_area:
            continue
        motion_boxes.append((x,y,w,h))
        img = cv2.rectangle(img,(x,y),(x+w,y+h),(255,0,0),1)
    cv2.imshow("bounding boxes for frame {frame_idx}",img)
    cv2.waitKey(0)
    return motion_boxes


In [ ]:
# Step 2: Detect motion in frames
'''import os
output_dir = os.path.join(os.getcwd(),"output_video")
frames_dir = os.path.join(output_dir, "results")
os.makedirs(frames_dir, exist_ok=True)'''
motion_results = []
frames = frames.copy()
for i, frame in enumerate(frames[0:2]):
    if i==1:
        cv2.imshow("frame1",frame)
    print(f"Processing frame {i + 1}/{len(frames)}")

    # Pass the entire frames list and the current index to detect_motion
    motion_boxes = detect_motion(frames, i)
    motion_results.append(motion_boxes)

Processing frame 1/50
Processing frame 2/50


In [ ]:
# viewport_tracker.py
"""
Viewport tracking functions for creating a smooth "virtual camera".
"""

import cv2
import numpy as np


def calculate_region_of_interest(motion_boxes, frame_shape):
    """
    Calculate the primary region of interest based on motion boxes.

    Args:
        motion_boxes: List of motion detection bounding boxes
        frame_shape: Shape of the video frame (height, width)

    Returns:
        Tuple (x, y, w, h) representing the region of interest center point and dimensions
    """
    # TODO: Implement region of interest calculation
    # 1. Choose a strategy for determining the main area of interest
    #    - You could use the largest motion box
    #    - Or combine nearby boxes
    #    - Or use a weighted average of all motion boxes
    # 2. Return the coordinates of the chosen region

    # Example starter code:
    if not motion_boxes:
        # If no motion is detected, use the center of the frame
        height, width = frame_shape[:2]
        return (width // 2, height // 2, 0, 0)

    # Your implementation here

    return (0, 0, 0, 0)  # Placeholder


def track_viewport(frames, motion_results, viewport_size, smoothing_factor=0.3):
    """
    Track viewport position across frames with smoothing.

    Args:
        frames: List of video frames
        motion_results: List of motion detection results for each frame
        viewport_size: Tuple (width, height) of the viewport
        smoothing_factor: Factor for smoothing viewport movement (0-1)
                          Lower values create smoother movement

    Returns:
        List of viewport positions for each frame as (x, y) center coordinates
    """
    # TODO: Implement viewport tracking with smoothing
    # 1. For each frame, determine the region of interest based on motion_results
    # 2. Apply smoothing to avoid jerky movements
    #    - Use previous viewport positions to smooth the movement
    #    - Consider implementing a simple exponential moving average
    #    - Or a more advanced approach like Kalman filtering
    # 3. Ensure the viewport stays within the frame boundaries
    # 4. Return the list of viewport positions for all frames

    # Example starter code:
    viewport_positions = []

    # Initialize with center of first frame if available
    if frames:
        height, width = frames[0].shape[:2]
        prev_x, prev_y = width // 2, height // 2
    else:
        return []

    # Your implementation here

    return viewport_positions

In [ ]:
frames[0].shape[:2]

In [ ]:
def calculate_region_of_interest(motion_boxes, frame_shape):
    """
    Calculate the primary region of interest based on motion boxes.

    Args:
        motion_boxes: List of motion detection bounding boxes
        frame_shape: Shape of the video frame (height, width)

    Returns:
        Tuple (x, y, w, h) representing the region of interest center point and dimensions
    """
    # TODO: Implement region of interest calculation
    # 1. Choose a strategy for determining the main area of interest
    #    - You could use the largest motion box
    #    - Or combine nearby boxes
    #    - Or use a weighted average of all motion boxes
    # 2. Return the coordinates of the chosen region

    # Example starter code:
    if not motion_boxes:
        # If no motion is detected, use the center of the frame
        height, width = frame_shape[:2]
        return (width // 2, height // 2, 0, 0)

    # Your implementation here
    # using the largest motion box strategy 
    largest_motion_box = motion_boxes[0]
    largest_area = motion_boxes[0][2] * motion_boxes[0][3]
    for i,box in enumerate(motion_boxes):
        if box[2] * box[3] > largest_area:
            largest_motion_box = box
    (x,y,w,h)=(largest_motion_box[0]+(largest_motion_box[2])//2,largest_motion_box[1]+(largest_motion_box[3])//2,largest_motion_box[2],largest_motion_box[3])

    return (x,y,w,h)


In [ ]:
import numpy as np
from sklearn.cluster import KMeans

def clustering_strategy(bounding_boxes):
    # Step 1: Compute centers
    centers = np.array([
        (x + w/2, y + h/2) for (x, y, w, h) in bounding_boxes
    ])

    # Step 2: KMeans clustering
    n_clusters=4
    kmeans = KMeans(n_clusters=n_clusters, init='k-means++',n_init=20,max_iter=300,random_state=42)
    labels = kmeans.fit_predict(centers)


    # Step 3: Select cluster (here, the largest cluster)
    unique, counts = np.unique(labels, return_counts=True)
    
    largest_cluster = unique[np.argmax(counts)]
    

    # Get bounding boxes in the largest cluster
    selected_boxes = [
        bbox for bbox, label in zip(bounding_boxes, labels) if label == largest_cluster
    ]

    # Step 4: Compute the final bounding box
    x_min = min(x for (x, y, w, h) in selected_boxes)
    y_min = min(y for (x, y, w, h) in selected_boxes)
    x_max = max(x + w for (x, y, w, h) in selected_boxes)
    y_max = max(y + h for (x, y, w, h) in selected_boxes)


    wfinal = x_max - x_min
    hfinal = y_max - y_min
    
    padding_ratio =0.1 # giving some padding to the region of interest so that roi is not too tight
    
    # add padding
    
    pad_w = int(round(wfinal * padding_ratio))
    pad_h = int(round(hfinal * padding_ratio))
    xfinal = x_min - pad_w//2
    yfinal = y_min - pad_h//2
    wfinal = wfinal + pad_w
    hfinal = hfinal + pad_h
    

    return (xfinal, yfinal, wfinal, hfinal)


In [ ]:
def track_viewport(frames, motion_results, viewport_size, smoothing_factor=0.3):
    """
    Track viewport position across frames with smoothing.

    Args:
        frames: List of video frames
        motion_results: List of motion detection results for each frame
        viewport_size: Tuple (width, height) of the viewport
        smoothing_factor: Factor for smoothing viewport movement (0-1)
                          Lower values create smoother movement

    Returns:
        List of viewport positions for each frame as (x, y) center coordinates
    """
    # TODO: Implement viewport tracking with smoothing
    # 1. For each frame, determine the region of interest based on motion_results
    # 2. Apply smoothing to avoid jerky movements
    #    - Use previous viewport positions to smooth the movement
    #    - Consider implementing a simple exponential moving average
    #    - Or a more advanced approach like Kalman filtering
    # 3. Ensure the viewport stays within the frame boundaries
    # 4. Return the list of viewport positions for all frames

    # Example starter code:
    viewport_positions = []

    # Initialize with center of first frame if available
    if frames:
        height, width = frames[0].shape[:2]
        prev_x, prev_y = width // 2, height // 2
    else:
        return []

    # Your implementation here
    w_viewport, h_viewport = viewport_size
    w_frame, h_frame = frames[0].shape[:2]
    
    # initialize kalman filter
    kalman = cv2.KalmanFilter(4,2)
    
    kalman.transitionMatrix = np.array([
        [1, 0, 1, 0],
        [0, 1, 0, 1],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ], np.float32)
    
    kalman.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
    kalman.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1
    kalman.statePre = np.array([[prev_x], [prev_y], [0], [0]], np.float32)
    
    for i,motion_boxes in enumerate(motion_results):
        
        # Region of Interest
        roi_x, roi_y, roi_w, roi_h = clustering_strategy(motion_boxes) # it gives (x,y,w,h)
        target_x = roi_x + roi_w / 2
        target_y = roi_y + roi_h / 2
        
        #predict
        prediction = kalman.predict()
        pred_x, pred_y = prediction[0, 0], prediction[1, 0]
        
        # Correct with new measurement
        measurement = np.array([[np.float32(target_x)], [np.float32(target_y)]])
        corrected = kalman.correct(measurement)
        corrected_x, corrected_y = corrected[0, 0], corrected[1, 0]
        
        # keep viewport inside frame
        half_vw, half_vh = w_viewport / 2, h_viewport / 2
        corrected_x = max(half_vw, min(w_frame - half_vw, corrected_x))
        corrected_y = max(half_vh, min(h_frame - half_vh, corrected_y))
        
        viewport_positions.append((corrected_x, corrected_y))
        
       
    return viewport_positions

In [256]:
min(1280-360,np.float32(725.6515))
max(360,np.float32(725.6515))

np.float32(725.6515)

In [ ]:
viewport_size = (720, 480)
viewport_positions = track_viewport(frames, motion_results[1:], viewport_size)

In [ ]:
import os
def visualize_results(frames, motion_results, viewport_positions, viewport_size, output_dir):
    """
    Create visualization of motion detection and viewport tracking results.

    Args:
        frames: List of video frames
        motion_results: List of motion detection results for each frame
        viewport_positions: List of viewport center positions for each frame
        viewport_size: Tuple (width, height) of the viewport
        output_dir: Directory to save visualization results
    """
    # Create output directory for frames
    frames_dir = os.path.join(output_dir, "frames")
    os.makedirs(frames_dir, exist_ok=True)

    viewport_dir = os.path.join(output_dir, "viewport")
    os.makedirs(viewport_dir, exist_ok=True)

    # Get dimensions for the output video
    height, width = frames[0].shape[:2]

    # Create video writers
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    video_path = os.path.join(output_dir, "motion_detection_kmeans4_5fps.mp4")
    video_writer = cv2.VideoWriter(video_path, fourcc, 5, (width, height))

    viewport_video_path = os.path.join(output_dir, "viewport_tracking_kmeans4_5fps.mp4")
    vp_width, vp_height = viewport_size
    viewport_writer = cv2.VideoWriter(
        viewport_video_path, fourcc, 5, (vp_width, vp_height)
    )

    # TODO: Implement visualization
    # 1. Process each frame
    #    a. Create a copy of the frame for visualization
    #    b. Draw bounding boxes around motion regions
    #       (hint: cv2.rectangle with green color (0, 255, 0))
    #    c. Draw the viewport rectangle
    #       (hint: cv2.rectangle with blue color (255, 0, 0))
    #    d. Extract the viewport content (the area inside the viewport)
    #    e. Add frame number to the visualization (hint: cv2.putText)
    #    f. Save visualization frames and viewport frames as images
    #    g. Write frames to both video writers
    # 2. Release the video writers when done

    # Example starter code:
    for i, frame in enumerate(frames):
        frame_copy = frame.copy()
        #Draw bounding boxes around motion regions
        motion_boxes=motion_results[i]
        for box in motion_boxes:
            cv2.rectangle(frame_copy,(box[0],box[1]),(box[0]+box[2],box[1]+box[3]),(0,255,0),1)
        # Draw view port rectangle 
        (x,y) = viewport_positions[i]
        x1 = int(x - vp_width/2)
        y1 = int(y - vp_height/2)
        x2 = int(x + vp_width/2)
        y2 = int(y + vp_height/2)
        cv2.rectangle(frame_copy, (x1, y1), (x2, y2), (255, 0, 0), 1)
        
        # extract the viewport content
        viewport_frame = frame_copy[y1:y2, x1:x2]
        
        # saving images
        filename = os.path.join(frames_dir, f"frame_{i+1:04d}.png")
        cv2.imwrite(filename, frame_copy)
        
        filename = os.path.join(viewport_dir, f"frame_{i+1:04d}.png")
        cv2.imwrite(filename, viewport_frame)      
        
        # writing frames to video writers
        video_writer.write(frame_copy)
        viewport_writer.write(viewport_frame)
        
    video_writer.release()
    viewport_writer.release()

    print(f"Visualization saved to {video_path}")
    print(f"Viewport video saved to {viewport_video_path}")
    print(f"Individual frames saved to {frames_dir} and {viewport_dir}")

In [ ]:
import os
video_path = os.path.join(os.getcwd(),"output_video")
video_path

In [ ]:
visualize_results(
        frames[1:], motion_results, viewport_positions, viewport_size, video_path
    )

In [254]:
1280-360

920

In [261]:
a= np.array([[1,2],[3,4],[5,6],[8,9]])

In [262]:
len(a)

4